In [ ]:

import warnings
warnings.filterwarnings('ignore')


import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel
import numpy as np
import os
from tqdm import tqdm
import gc
import json
import random
from datetime import datetime
warnings.filterwarnings('ignore')


print("=" * 80)
print("Loading datasets...")
print("=" * 80)

train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")
tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()

print(f"✓ Training samples: {len(train_data)}")
print(f"✓ Testing samples: {len(test_data)}")
print(f"✓ Languages: {len(tag_vocab)}")
print(f"Languages: {tag_vocab}")


print("\n" + "=" * 80)
print("Loading Mistral tokenizer...")
print("=" * 80)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("✓ Mistral tokenizer loaded successfully!")
except Exception as e:
    print(f"✗ Error loading Mistral: {e}")
    print("Falling back to smaller model...")
    MODEL_NAME = "microsoft/codebert-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"✓ Loaded fallback: {MODEL_NAME}")


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Tokenizer config: pad_token={tokenizer.pad_token}, padding_side={tokenizer.padding_side}")


def tokenize_batch(texts, max_length=256):
    """Tokenize code snippets"""
    return tokenizer(
        texts,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

print("\nTokenizing training data...")
train_texts = train_data["code"].tolist()
train_encodings = tokenize_batch(train_texts, max_length=256)

print("Tokenizing test data...")
test_texts = test_data["code"].tolist()
test_encodings = tokenize_batch(test_texts, max_length=256)


class CodeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }


class MistralClassifier(nn.Module):
    def __init__(self, num_classes, model_name=MODEL_NAME, device=None):
        super().__init__()
        
        print(f"Loading {model_name}...")
        

        self.encoder = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if "mistral" in model_name.lower() else torch.float32
        )
        
        for param in self.encoder.parameters():
            param.requires_grad = False
        
        hidden_size = self.encoder.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        
        if device:
            self.encoder = self.encoder.to(device)
            self.classifier = self.classifier.to(device)
        
        print(f"✓ Model loaded successfully!")
        print(f"  Hidden size: {hidden_size}")
        print(f"  Num classes: {num_classes}")
        print(f"  Trainable params: {sum(p.numel() for p in self.classifier.parameters()):,}")
    
    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.long()
        attention_mask = attention_mask.float()
        
        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        
        if last_hidden.dtype != mask.dtype:
            last_hidden = last_hidden.to(mask.dtype)
        
        sum_embeddings = torch.sum(last_hidden * mask, dim=1)
        sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        
        pooled = pooled.float()
        
        logits = self.classifier(pooled)
        return logits

def train_epoch(model, data_loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(data_loader), correct / total

def evaluate_model(model, data_loader, device, tag_vocab):
    """Evaluate model and return detailed results"""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.extend(probs.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    per_lang_results = {}
    for i, lang in enumerate(tag_vocab):
        lang_indices = np.where(np.array(all_labels) == i)[0]
        if len(lang_indices) > 0:
            lang_acc = accuracy_score(
                np.array(all_labels)[lang_indices],
                np.array(all_preds)[lang_indices]
            )
            per_lang_results[lang] = {
                'accuracy': lang_acc,
                'samples': len(lang_indices)
            }
    
    return accuracy, all_preds, all_labels, all_probs, per_lang_results

def run_mistral_experiment(seed, device, tag_vocab, train_data, test_data, 
                          train_encodings, test_encodings, save_results=True):
    """Run a single Mistral experiment with given seed"""
    print(f"\n{'='*80}")
    print(f"MISTRAL EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*80}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    label_encoder = LabelEncoder()
    label_encoder.fit(tag_vocab)
    
    train_dataset = CodeDataset(
        train_encodings,
        label_encoder.transform(train_data["language"])
    )
    
    test_dataset = CodeDataset(
        test_encodings,
        label_encoder.transform(test_data["language"])
    )
    
    num_classes = len(tag_vocab)
    model = MistralClassifier(num_classes, MODEL_NAME, device)
    
    optimizer = optim.AdamW(
        model.classifier.parameters(),
        lr=1e-3,
        weight_decay=0.01
    )
    
    criterion = nn.CrossEntropyLoss()
    
    batch_size = 8 if "mistral" in MODEL_NAME.lower() else 16
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\nTraining configuration:")
    print(f"  Seed: {seed}")
    print(f"  Model: {MODEL_NAME}")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {1e-3}")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Test samples: {len(test_dataset)}")
    
    print(f"\n{'='*50}")
    print(f"Training for seed {seed}")
    print(f"{'='*50}")
    
    epochs = 3
    best_accuracy = 0
    training_history = []
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 40)
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        
        val_accuracy, val_preds, val_labels, val_probs, val_per_lang = evaluate_model(
            model, test_loader, device, tag_vocab
        )
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        
        training_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_acc': val_accuracy
        })
        
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_train_acc = train_acc
            best_train_loss = train_loss
    
    print(f"\n{'='*40}")
    print(f"Final Evaluation for seed {seed}")
    print(f"{'='*40}")
    
    model.eval()
    final_accuracy, all_preds, all_labels, all_probs, per_lang_results = evaluate_model(
        model, test_loader, device, tag_vocab
    )
    
    confidences = np.max(all_probs, axis=1)
    
    print(f"\n✓ Experiment completed for seed {seed}!")
    print(f"  Final Test Accuracy: {final_accuracy:.4f}")
    print(f"  Training Accuracy: {best_train_acc:.4f}")
    print(f"  Training Loss: {best_train_loss:.4f}")
    
    if save_results:
        model_save_path = f"/home/aman_swaraj/Downloads/Codelite/mistral_seed{seed}.pth"
        torch.save({
            'seed': seed,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'final_accuracy': final_accuracy,
            'train_accuracy': best_train_acc,
            'train_loss': best_train_loss,
            'tag_vocab': tag_vocab,
            'label_encoder_classes': label_encoder.classes_.tolist(),
            'model_name': MODEL_NAME
        }, model_save_path)
        print(f"✓ Model saved to {model_save_path}")
        
        predictions_df = pd.DataFrame({
            'true_label': [tag_vocab[l] for l in all_labels],
            'predicted_label': [tag_vocab[p] for p in all_preds],
            'confidence': confidences,
            'is_correct': [1 if p == l else 0 for p, l in zip(all_preds, all_labels)]
        })
        
        results_path = f"/home/aman_swaraj/Downloads/Codelite/mistral_results_seed{seed}.csv"
        predictions_df.to_csv(results_path, index=False)
        print(f"✓ Results saved to {results_path}")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'seed': seed,
        'final_accuracy': final_accuracy,
        'train_accuracy': best_train_acc,
        'train_loss': best_train_loss,
        'per_language_accuracy': per_lang_results,
        'predictions': all_preds,
        'labels': all_labels,
        'confidences': confidences
    }

def main():
    print("\n" + "=" * 80)
    print("MULTI-SEED MISTRAL EXPERIMENT")
    print("=" * 80)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        
        torch.cuda.empty_cache()
    
    seeds = [42, 123, 456, 789, 999]  
    print(f"\nRunning experiments for {len(seeds)} seeds: {seeds}")
    
    all_results = []
    
    for i, seed in enumerate(seeds):
        print(f"\n{'#'*80}")
        print(f"Experiment {i+1}/{len(seeds)}")
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'#'*80}")
        
        try:
            result = run_mistral_experiment(
                seed=seed,
                device=device,
                tag_vocab=tag_vocab,
                train_data=train_data,
                test_data=test_data,
                train_encodings=train_encodings,
                test_encodings=test_encodings,
                save_results=True
            )
            
            all_results.append(result)
            
            print(f"\n✓ Experiment {i+1} completed successfully!")
            
        except Exception as e:
            print(f"\n✗ Experiment {i+1} failed with error:")
            print(f"  Error: {e}")
            print(f"  Skipping seed {seed}...")
            continue
    
    if not all_results:
        print("\nNo experiments completed successfully!")
        return
    
    print("\n" + "=" * 80)
    print("SUMMARY OF ALL MISTRAL EXPERIMENTS")
    print("=" * 80)
    
    final_accuracies = [r['final_accuracy'] for r in all_results]
    train_accuracies = [r['train_accuracy'] for r in all_results]
    train_losses = [r['train_loss'] for r in all_results]
    seeds_list = [r['seed'] for r in all_results]
    
    summary_data = []
    for i, result in enumerate(all_results):
        summary_data.append({
            'Seed': result['seed'],
            'Final_Accuracy': result['final_accuracy'],
            'Final_Accuracy_%': f"{result['final_accuracy']*100:.2f}%",
            'Train_Accuracy': result['train_accuracy'],
            'Train_Loss': result['train_loss']
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    if len(final_accuracies) > 0:
        mean_accuracy = np.mean(final_accuracies)
        std_accuracy = np.std(final_accuracies)
        min_accuracy = np.min(final_accuracies)
        max_accuracy = np.max(final_accuracies)
    else:
        mean_accuracy = std_accuracy = min_accuracy = max_accuracy = 0
    
    print(f"\nOVERALL STATISTICS:")
    print(f"  Number of experiments: {len(all_results)}")
    print(f"  Mean Final Accuracy: {mean_accuracy:.4f} ({mean_accuracy*100:.2f}%)")
    print(f"  Std Final Accuracy: {std_accuracy:.4f}")
    print(f"  Min Final Accuracy: {min_accuracy:.4f} ({min_accuracy*100:.2f}%)")
    print(f"  Max Final Accuracy: {max_accuracy:.4f} ({max_accuracy*100:.2f}%)")
    print(f"  Range: {max_accuracy - min_accuracy:.4f}")
    
    print(f"\nDETAILED RESULTS:")
    print(summary_df.to_string(index=False))
    
    if len(all_results) > 0:
        print(f"\n{'='*80}")
        print("LANGUAGE-WISE ANALYSIS (Across all seeds)")
        print(f"{'='*80}")
        
        lang_accuracies = {lang: [] for lang in tag_vocab}
        lang_samples = {lang: [] for lang in tag_vocab}
        
        for result in all_results:
            for lang, lang_data in result['per_language_accuracy'].items():
                lang_accuracies[lang].append(lang_data['accuracy'])
                lang_samples[lang].append(lang_data['samples'])
        
        lang_summary = []
        for lang in tag_vocab:
            if lang_accuracies[lang]:  
                mean_acc = np.mean(lang_accuracies[lang])
                std_acc = np.std(lang_accuracies[lang])
                min_acc = np.min(lang_accuracies[lang])
                max_acc = np.max(lang_accuracies[lang])
                avg_samples = np.mean(lang_samples[lang]) if lang_samples[lang] else 0
                
                lang_summary.append({
                    'Language': lang,
                    'Mean_Accuracy': mean_acc,
                    'Std_Accuracy': std_acc,
                    'Min_Accuracy': min_acc,
                    'Max_Accuracy': max_acc,
                    'Range': max_acc - min_acc,
                    'Avg_Samples': int(avg_samples)
                })
        
        if lang_summary:
            lang_summary.sort(key=lambda x: x['Mean_Accuracy'], reverse=True)
            lang_df = pd.DataFrame(lang_summary)
            
            print("\nLanguage-wise performance across seeds (sorted by mean accuracy):")
            print(lang_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
            
            lang_mean_accuracies = [lang['Mean_Accuracy'] for lang in lang_summary]
            print(f"\nLanguage-wise Statistics:")
            print(f"  Mean accuracy across languages: {np.mean(lang_mean_accuracies):.4f}")
            print(f"  Std of language accuracies: {np.std(lang_mean_accuracies):.4f}")
            print(f"  Best performing language: {lang_summary[0]['Language']} ({lang_summary[0]['Mean_Accuracy']:.4f})")
            print(f"  Worst performing language: {lang_summary[-1]['Language']} ({lang_summary[-1]['Mean_Accuracy']:.4f})")
    
    if len(all_results) > 0:
        print(f"\n{'='*80}")
        print("CONFIDENCE ANALYSIS (Across all seeds)")
        print(f"{'='*80}")
        
        all_confidences = []
        for result in all_results:
            all_confidences.extend(result['confidences'])
        
        print(f"Overall Confidence Statistics:")
        print(f"  Average confidence: {np.mean(all_confidences):.4f}")
        print(f"  Confidence std: {np.std(all_confidences):.4f}")
        print(f"  Min confidence: {np.min(all_confidences):.4f}")
        print(f"  Max confidence: {np.max(all_confidences):.4f}")
        
        if len(all_results) > 1:
            confidence_means = [np.mean(r['confidences']) for r in all_results]
            accuracy_correlation = np.corrcoef(confidence_means, final_accuracies)[0, 1]
            print(f"  Correlation (confidence vs accuracy): {accuracy_correlation:.4f}")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_results:
        detailed_summary = []
        for result in all_results:
            row = {
                'seed': result['seed'],
                'final_accuracy': result['final_accuracy'],
                'train_accuracy': result['train_accuracy'],
                'train_loss': result['train_loss']
            }
            
            for lang in tag_vocab:
                if lang in result['per_language_accuracy']:
                    row[f'acc_{lang}'] = result['per_language_accuracy'][lang]['accuracy']
                else:
                    row[f'acc_{lang}'] = np.nan
            
            detailed_summary.append(row)
        
        detailed_df = pd.DataFrame(detailed_summary)
        summary_path = f"/home/aman_swaraj/Downloads/Codelite/mistral_multiseed_summary_{timestamp}.csv"
        detailed_df.to_csv(summary_path, index=False)
        print(f"\n✓ Detailed summary saved to: {summary_path}")
        
        if 'lang_df' in locals():
            lang_summary_path = f"/home/aman_swaraj/Downloads/Codelite/mistral_lang_summary_{timestamp}.csv"
            lang_df.to_csv(lang_summary_path, index=False)
            print(f"✓ Language-wise summary saved to: {lang_summary_path}")
    
    stats_summary = {
        'timestamp': timestamp,
        'num_experiments': len(all_results),
        'seeds': seeds_list,
        'mean_final_accuracy': mean_accuracy,
        'std_final_accuracy': std_accuracy,
        'min_final_accuracy': min_accuracy,
        'max_final_accuracy': max_accuracy,
        'accuracy_range': max_accuracy - min_accuracy,
        'mean_train_accuracy': np.mean(train_accuracies) if train_accuracies else 0,
        'mean_train_loss': np.mean(train_losses) if train_losses else 0,
        'mean_confidence': np.mean(all_confidences) if 'all_confidences' in locals() else 0,
        'model_name': MODEL_NAME
    }
    
    stats_df = pd.DataFrame([stats_summary])
    stats_path = f"/home/aman_swaraj/Downloads/Codelite/mistral_stats_summary_{timestamp}.csv"
    stats_df.to_csv(stats_path, index=False)
    print(f"✓ Statistics summary saved to: {stats_path}")
    
    print("\n" + "=" * 80)
    print("FINAL REPORT")
    print("=" * 80)
    
    print(f"\nModel: {MODEL_NAME}")
    print(f"Training samples: {len(train_data)}")
    print(f"Test samples: {len(test_data)}")
    print(f"Number of languages: {len(tag_vocab)}")
    print(f"Number of experiments: {len(all_results)}")
    
    if len(all_results) > 0:
        print(f"\nPerformance Summary:")
        print(f"  Overall mean accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
        print(f"  Best seed: {seeds_list[np.argmax(final_accuracies)]} ({max_accuracy:.4f})")
        print(f"  Worst seed: {seeds_list[np.argmin(final_accuracies)]} ({min_accuracy:.4f})")
        
        if len(all_results) > 1 and 'lang_summary' in locals():
            print(f"\nTop 5 Languages:")
            for i in range(min(5, len(lang_summary))):
                lang = lang_summary[i]
                print(f"  {i+1}. {lang['Language']}: {lang['Mean_Accuracy']:.4f} ± {lang['Std_Accuracy']:.4f}")
            
            print(f"\nBottom 5 Languages:")
            for i in range(1, min(6, len(lang_summary))):
                lang = lang_summary[-i]
                print(f"  {i}. {lang['Language']}: {lang['Mean_Accuracy']:.4f} ± {lang['Std_Accuracy']:.4f}")
        
        print(f"\nVariability Analysis:")
        print(f"  Overall accuracy range: {max_accuracy - min_accuracy:.4f}")
        if 'lang_summary' in locals():
            avg_lang_range = np.mean([lang['Range'] for lang in lang_summary])
            print(f"  Average language accuracy range: {avg_lang_range:.4f}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"\nGPU memory cleared.")
    
    print("\n" + "=" * 80)
    print("MISTRAL MULTI-SEED EXPERIMENT COMPLETE!")
    print("=" * 80)

if __name__ == "__main__":
    main()